In [14]:
puts `head -2 ./maps/genes.map`

sourceid,label,geneid,protein,recommended_full,taxon
ENSG00000012779,ENSG00000012779,http://purl.uniprot.org/geneid/240,http://purl.uniprot.org/uniprot/P09917,Polyunsaturated fatty acid 5-lipoxygenase,http://purl.uniprot.org/taxonomy/9606


In [5]:
puts `head -2 ./maps/diseases.map`

source,mondo,prefname
EFO_0000174,http://purl.obolibrary.org/obo/MONDO_0012817,Ewing sarcoma


In [4]:
puts # `grep Orphanet_22 ./maps/diseases.map | head -20`

In [6]:
puts `head -5 ./rawdata/disease-gene.csv`

"source","source_type","target","target_type"
"DOID_7551","disease","ENSG00000058085","gene"
"DOID_7551","disease","ENSG00000091831","gene"
"DOID_7551","disease","ENSG00000102755","gene"
"DOID_7551","disease","ENSG00000105329","gene"


In [7]:
require 'net/http'
require 'json'
require 'uri'

# Fetch the rdfs:label for an HPO term via EBI OLS4 (authoritative mirror, always current).
# hp_code: "HP:0001250" or "HP_0001250"
require 'net/http'
require 'json'
require 'uri'

def get_hpo_label(hp_code)
  local   = hp_code.tr(':', '_').then { |s| s.start_with?('HP_') ? s : "HP_#{s}" }
  iri     = "http://purl.obolibrary.org/obo/#{local}"
  encoded = URI.encode_uri_component(URI.encode_uri_component(iri))
  uri     = URI("https://www.ebi.ac.uk/ols4/api/ontologies/hp/terms/#{encoded}")

  response = Net::HTTP.get_response(uri)
  return "no HPO match found for #{local}" unless response.is_a?(Net::HTTPSuccess)

  data = JSON.parse(response.body)
  data['label'] || "no HPO match found for #{local}"
rescue => e
  "no HPO match found for #{local}"
end

puts get_hpo_label('HP:0001250')   # => "Seizure"
puts get_hpo_label('HP_0001250')   # also works

Seizure
Seizure


In [10]:
require 'linkeddata'
require 'rdf/nquads'
require 'csv'

e = File.open('./graph/phenotype-gene-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')
# Read input files

gene_mappings = CSV.read('./maps/genes.map', headers: true)
failures = {}


# refresh
f = File.open('./graph/radboud_phenotype-gene.nq.large', 'w')
f.close

# diseases includes phenotypes
CSV.foreach('./rawdata/disease-gene.csv', col_sep: ",", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
# "source","source_type","target","target_type"
# "DOID_7551","disease","ENSG00000058085","gene"      
  hpo_num = row['source']
  next unless hpo_num.match(/HP_/)  # phenos only
  # warn "FOUND: #{hpo_num}"
  
  gene_id = row['target']  
    # gene mappings
    #  sourceid,label,geneid,protein,recommended_full,taxon
    # ENSG00000012779,ENSG00000012779,http://purl.uniprot.org/geneid/240,http://purl.uniprot.org/uniprot/P09917,Polyunsaturated fatty acid 5-lipoxygenase,http://purl.uniprot.org/taxonomy/9606
  gene = gene_mappings.find { |d| d['sourceid'] == gene_id }
  unless gene
    next if failures[gene_id]
    failures[gene_id] = 1
    warn "gene lookup failed #{gene_id}"
    e.write "gene lookup failed #{gene_id}\n"
    next
  end
  

  hpo      = RDF::URI.new("http://purl.obolibrary.org/obo/#{hpo_num}")
  hpo_type      = RDF::URI.new("http://edamontology.org/data_3275")
  hpo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Phenotype") 
  hpo_label = RDF::Literal.new(get_hpo_label(hpo_num))
  original_disease = RDF::Literal.new(hpo_num)
  

#   sourceid,label,geneid,protein,recommended_full,taxon
#   ENSG00000091831,ENSG00000091831,http://purl.uniprot.org/geneid/2099,http://purl.uniprot.org/uniprot/P03372,Estrogen receptor,http://purl.uniprot.org/taxonomy/9606
  gene_uri = RDF::URI.new(gene['geneid'])
  gene_type = RDF::URI.new("http://edamontology.org/data_1027")
  gene_label =  RDF::Literal.new("NCBI/UniProt Gene Identifier")
  gene_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Gene")
  gene_label = RDF::Literal.new(gene['label'])

  
  protein_uri = RDF::URI.new(gene['protein'])
  protein_type = RDF::URI.new("http://edamontology.org/data_2291")
  protein_label =  RDF::Literal.new("UniProt Identifier")
  protein_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Protein")
  human_protein_label = RDF::Literal.new(gene['recommended_full'])
    
  taxon = RDF::URI.new(gene['taxon'])
  
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:#{hpo_num}_#{gene_id}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
  
  # Create RDF repository (need to do this each time, since there are hundreds of thousands of lines, and the graph gets too big for memory)
  graph = RDF::Repository.new


      
  # Add quads to graph using RDF::Statement
  graph << RDF::Statement.new(hpo, SIMPATHIC['associated-with'], protein_uri, graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri, SIMPATHIC['associated-with'], hpo, graph_name: context_uri)
  graph << RDF::Statement.new(hpo, SIMPATHIC['associated-with'], gene_uri, graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri, SIMPATHIC['associated-with'], hpo, graph_name: context_uri)

    graph << RDF::Statement.new(hpo,          RDFS.label,               hpo_label,                                    graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDF.type,                 hpo_type,                                     graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          RDF.type,                 hpo_core_type,                                graph_name: context_uri)
    graph << RDF::Statement.new(hpo_type,     RDFS.label,               RDF::Literal.new("HPO Ontology Term"),        graph_name: context_uri)
    graph << RDF::Statement.new(hpo_core_type,RDFS.label,               RDF::Literal.new("Phenotype"),                graph_name: context_uri)
    graph << RDF::Statement.new(hpo,          SIMPATHIC['original-id'], original_disease,                             graph_name: context_uri)

  graph << RDF::Statement.new(gene_uri,  RDFS.label,       gene_label , graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_type, graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(gene_type, RDFS.label,       RDF::Literal.new("NCBI Gene"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_core_type, RDFS.label,  RDF::Literal.new("Gene"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)
    
  graph << RDF::Statement.new(protein_uri,  RDFS.label,       human_protein_label , graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_type, graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(protein_type, RDFS.label,       RDF::Literal.new("UniProt"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_core_type, RDFS.label,  RDF::Literal.new("Protein"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)

  
    graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'],       RDF::Literal.new("Radboud"),     graph_name: general_context)
    # graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'],         RDF::URI.new(evidence),                   graph_name: general_context)
    # graph << RDF::Statement.new(context_uri, SIMPATHIC['score'],            RDF::Literal.new(score),                  graph_name: general_context)
    graph << RDF::Statement.new(context_uri, SIMPATHIC['source-relation'],  RDF::Literal.new("ASSOCIATED_WITH"),      graph_name: general_context)


#   warn "graph #{context_uri} built"
  # Write RDF to file in N-Quads format
  File.open('./graph/radboud_phenotype-gene.nq.large', 'a') do |f|
    RDF::Writer.for(:nquads).new(f) do |writer|
#         warn "writing quads"
      writer << graph
    end
  end
#   warn "end graph writing"

end
warn "completed graph building"
e.close

puts "RDF quads written"

(irb):7: warning: already initialized constant Object::SIMPATHIC
(irb):7: warning: previous definition of SIMPATHIC was here
(irb):8: warning: already initialized constant Object::RDFS
(irb):8: warning: previous definition of RDFS was here
gene lookup failed ENSG00000123201
gene lookup failed ENSG00000211891
completed graph building


RDF quads written


In [11]:
puts `cat ./graph/radboud_phenotype-gene.nq.large | wc -l`
puts `head -40 ./graph/radboud_phenotype-gene.nq.large`

105170
<http://purl.obolibrary.org/obo/HP_0000857> <urn:simpathic:associated-with> <http://purl.uniprot.org/uniprot/Q14654> <urn:simpathic:context:HP_0000857_ENSG00000187486> .
<http://purl.obolibrary.org/obo/HP_0000857> <urn:simpathic:associated-with> <http://purl.uniprot.org/geneid/3767> <urn:simpathic:context:HP_0000857_ENSG00000187486> .
<http://purl.obolibrary.org/obo/HP_0000857> <http://www.w3.org/2000/01/rdf-schema#label> "Neonatal insulin-dependent diabetes mellitus" <urn:simpathic:context:HP_0000857_ENSG00000187486> .
<http://purl.obolibrary.org/obo/HP_0000857> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://edamontology.org/data_3275> <urn:simpathic:context:HP_0000857_ENSG00000187486> .
<http://purl.obolibrary.org/obo/HP_0000857> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <https://w3id.org/biolink/vocab/Phenotype> <urn:simpathic:context:HP_0000857_ENSG00000187486> .
<http://purl.obolibrary.org/obo/HP_0000857> <urn:simpathic:original-id> "HP_0000857" <urn:simp